# 02 : Workspace Modules Lab

Main lesson:

```text
Notebook
→ runtime control, display, orchestration shell

src/
→ reusable transformation and DQ logic
```


In [0]:
import sys
import os
from pathlib import Path

marker = "RETAIL_LAB.json"
cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

for candidate in [cwd, *cwd.parents]:
    if (candidate / marker).exists():
        PROJECT_ROOT = str(candidate)
        break

if PROJECT_ROOT is None:
    for search_root in [Path("/Workspace/Users"), Path("/Workspace/Shared"), Path("/Workspace")]:
        if not search_root.exists():
            continue
        for current_root, dirs, files in os.walk(search_root):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in {"Trash", ".Trash"}]
            if marker in files:
                PROJECT_ROOT = current_root
                break
        if PROJECT_ROOT:
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project not found. Import the complete ZIP into Databricks Workspace."
    )

SRC_ROOT = f"{PROJECT_ROOT}/src"
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

from bootstrap import ensure_lab

ctx = ensure_lab(
    spark=spark,
    project_root=PROJECT_ROOT,
)

print("Project root :", PROJECT_ROOT)
print("Catalog      :", ctx.catalog)
print("Schema       :", ctx.schema)
print("Volume       :", ctx.volume)
print("Source root  :", ctx.source_root)


In [0]:
from schemas import ORDERS_SCHEMA
from readers import read_csv
from transforms import clean_orders, filter_orders
from quality import duplicate_keys, require_zero_rows

orders_raw_df = read_csv(
    spark,
    f"{ctx.source_root}/orders.csv",
    ORDERS_SCHEMA,
)
orders_clean_df = clean_orders(orders_raw_df)

assert orders_clean_df.count() == 126


In [0]:
dbutils.widgets.dropdown(
    "lab_order_status", "DELIVERED",
    ["DELIVERED", "SHIPPED", "PLACED", "CANCELLED"],
    "Lab Order Status"
)
dbutils.widgets.text(
    "lab_shipping_city", "", "Lab Shipping City"
)

lab_order_status = dbutils.widgets.get("lab_order_status")
lab_shipping_city = dbutils.widgets.get("lab_shipping_city")

orders_filtered_df = filter_orders(
    orders_clean_df,
    lab_order_status,
    lab_shipping_city,
)

print("Filtered rows:", orders_filtered_df.count())
display(orders_filtered_df.limit(30))


In [0]:
duplicate_order_ids_df = duplicate_keys(
    orders_clean_df, "order_id"
)
require_zero_rows(
    duplicate_order_ids_df,
    "duplicate order_id"
)
print("DQ PASS: order_id unique")
